# CALF BTCUSDT Pred16 Checkpoint Evaluation

This notebook evaluates existing CALF checkpoints for `BTCUSDT_96_16_*` without modifying the author's source files. It imports CALF modules, loads each checkpoint, collects predictions and labels from the test split, then visualizes how the models forecast the next 16 candles.

Run with the `llm-ts` environment/kernel that contains PyTorch, Transformers, einops, pandas, matplotlib, and the CALF dependencies.


## References And Design Choices

- CALF uses GPT-2 with `inputs_embeds`; Hugging Face documents `inputs_embeds` as a supported `GPT2Model` forward argument.
- Checkpoints are loaded through CALF's `load_state_dict_checkpoint`, which wraps PyTorch `torch.load(..., weights_only=True)` behavior.
- The notebook is structured as ordered Markdown and code cells, following the Jupyter notebook JSON model.

Sources: Hugging Face GPT-2 docs, PyTorch `torch.load` docs, and Jupyter `nbformat` docs.


In [1]:
from __future__ import annotations

import gc
import os
import re
import sys
import time
from argparse import Namespace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("matplotlib", "inline")
except Exception as exc:
    print(f"matplotlib inline setup skipped: {exc}")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

SEED = 2021
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
print("cuda_count", torch.cuda.device_count())
if torch.cuda.is_available():
    print("cuda_device", torch.cuda.get_device_name(0))


torch 2.7.1+cu118
cuda_available True
cuda_count 1
cuda_device NVIDIA GeForce RTX 2050


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
CALF_ROOT = REPO_ROOT / "CALF"
DATA_FILE = CALF_ROOT / "dataset" / "clean" / "BTCUSDT_15m_calf.csv"
CHECKPOINT_ROOT = CALF_ROOT / "checkpoints"
WTE_PATH = CALF_ROOT / "wte_pca_500.pt"

assert CALF_ROOT.exists(), CALF_ROOT
assert DATA_FILE.exists(), DATA_FILE
assert CHECKPOINT_ROOT.exists(), CHECKPOINT_ROOT
assert WTE_PATH.exists(), WTE_PATH

if str(CALF_ROOT) not in sys.path:
    sys.path.insert(0, str(CALF_ROOT))
os.chdir(CALF_ROOT)

print("repo", REPO_ROOT)
print("calf", CALF_ROOT)
print("data", DATA_FILE)


repo C:\Users\USER\Desktop\time_series\LLM for timeseries\LLMsForTimeSeries
calf C:\Users\USER\Desktop\time_series\LLM for timeseries\LLMsForTimeSeries\CALF
data C:\Users\USER\Desktop\time_series\LLM for timeseries\LLMsForTimeSeries\CALF\dataset\clean\BTCUSDT_15m_calf.csv


In [3]:
# Runtime knobs. Keep MAX_BATCHES=None for the full test split and author-comparable metrics.
# Set MAX_BATCHES=4 for a fast smoke run.
SELECTED_VARIANTS = ["ori", "dropAttn_keepWE", "llm_to_attn", "llm_to_trsf"]
SELECTED_ITERS = [0, 1]
MAX_BATCHES = None
STRICT_LOAD = True
DEVICE_OVERRIDE = "auto"  # "auto", "cpu", or "cuda:0"
REQUIRE_CUDA = True  # Fail early instead of silently falling back to CPU.

SEQ_LEN = 96
LABEL_LEN = 0
PRED_LEN = 16
BATCH_SIZE = 128
GPT_LAYERS = 6
D_MODEL = 768
N_HEADS = 8
D_FF = 768
ENC_IN = 7
C_OUT = 7
TARGET = "close"
FREQ = "15min"


In [4]:
ckpt_pattern = re.compile(
    r"^long_term_forecast_"
    r"(?P<model_id>BTCUSDT_96_16_(?P<variant>ori|dropAttn_keepWE|llm_to_attn|llm_to_trsf))"
    r"_GPT4TS_custom_ftM_sl96_ll0_pl16_dm768_nh8_el2_dl1_df768_fc1_ebtimeF_dtTrue_test_gpt6_(?P<itr>\d+)$"
)

records = []
for path in CHECKPOINT_ROOT.glob("long_term_forecast_BTCUSDT_96_16_*_gpt6_*/checkpoint.pth"):
    match = ckpt_pattern.match(path.parent.name)
    if not match:
        continue
    records.append(
        {
            "variant": match.group("variant"),
            "itr": int(match.group("itr")),
            "model_id": match.group("model_id"),
            "setting": path.parent.name,
            "checkpoint_path": path,
            "size_mb": path.stat().st_size / (1024**2),
        }
    )

inventory = pd.DataFrame(records).sort_values(["variant", "itr"]).reset_index(drop=True)
selected = inventory[
    inventory["variant"].isin(SELECTED_VARIANTS) & inventory["itr"].isin(SELECTED_ITERS)
].copy()

print(f"discovered checkpoints: {len(inventory)}")
print(f"selected checkpoints: {len(selected)}")
display(selected[["variant", "itr", "model_id", "size_mb", "setting"]])

if selected.empty:
    raise RuntimeError("No pred_len=16 checkpoints matched the selected filters.")


discovered checkpoints: 8
selected checkpoints: 8


,variant,itr,model_id,size_mb,setting
0,dropAttn_keepWE,0,BTCUSDT_96_16_dropAttn_keepWE,61.889760,long_term_forecast_BTCUSDT_96_16_dropAttn_keep...
1,dropAttn_keepWE,1,BTCUSDT_96_16_dropAttn_keepWE,61.889760,long_term_forecast_BTCUSDT_96_16_dropAttn_keep...
2,llm_to_attn,0,BTCUSDT_96_16_llm_to_attn,695.894120,long_term_forecast_BTCUSDT_96_16_llm_to_attn_G...
3,llm_to_attn,1,BTCUSDT_96_16_llm_to_attn,695.894120,long_term_forecast_BTCUSDT_96_16_llm_to_attn_G...
4,llm_to_trsf,0,BTCUSDT_96_16_llm_to_trsf,395.427257,long_term_forecast_BTCUSDT_96_16_llm_to_trsf_G...
5,llm_to_trsf,1,BTCUSDT_96_16_llm_to_trsf,395.427257,long_term_forecast_BTCUSDT_96_16_llm_to_trsf_G...
6,ori,0,BTCUSDT_96_16_ori,686.886058,long_term_forecast_BTCUSDT_96_16_ori_GPT4TS_cu...
7,ori,1,BTCUSDT_96_16_ori,686.886058,long_term_forecast_BTCUSDT_96_16_ori_GPT4TS_cu...


In [5]:
# Compatibility patch for loading CALF/wte_pca_500.pt with PyTorch 2.6+ weights_only=True.
# This keeps the author source untouched and only broadens the notebook runtime allowlist
# for the trusted local NumPy-backed WTE artifact.
import importlib

from torch.serialization import safe_globals


def numpy_torch_artifact_safe_globals() -> list:
    allowed = [np.ndarray, np.dtype]

    for dtype_name in [
        "bool",
        "float16",
        "float32",
        "float64",
        "int8",
        "int16",
        "int32",
        "int64",
        "uint8",
        "uint16",
        "uint32",
        "uint64",
    ]:
        allowed.append(type(np.dtype(dtype_name)))

    for module_name in ["numpy.core.multiarray", "numpy._core.multiarray"]:
        try:
            reconstruct = getattr(importlib.import_module(module_name), "_reconstruct")
        except Exception:
            continue
        allowed.append(reconstruct)
        allowed.append((reconstruct, f"{module_name}._reconstruct"))

    return allowed


def load_numpy_torch_artifact_notebook(path, map_location=None):
    with safe_globals(numpy_torch_artifact_safe_globals()):
        return torch.load(path, map_location=map_location, weights_only=True)


# Patch both the utility module and the GPT4TS module-level function imported from it.
import utils.torch_compat as torch_compat
import models.GPT4TS as gpt4ts_module

torch_compat.load_numpy_torch_artifact = load_numpy_torch_artifact_notebook
gpt4ts_module.load_numpy_torch_artifact = load_numpy_torch_artifact_notebook

# Smoke-test the WTE artifact before constructing the model.
wte_smoke = load_numpy_torch_artifact_notebook(WTE_PATH, map_location="cpu")
print("wte", tuple(wte_smoke.shape), wte_smoke.dtype)
del wte_smoke


c:\Users\USER\anaconda3\envs\llm-ts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


wte (768, 500) float32


In [6]:
from data_provider.data_factory import data_provider
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
from utils.metrics import metric as calf_metric
from utils.torch_compat import load_state_dict_checkpoint


def resolve_device() -> torch.device:
    cuda_available = torch.cuda.is_available()
    if DEVICE_OVERRIDE == "auto":
        if cuda_available:
            return torch.device("cuda:0")
        if REQUIRE_CUDA:
            raise RuntimeError(
                "GPU is required, but this notebook kernel is using a PyTorch build without CUDA. "
                f"torch={torch.__version__}, torch.version.cuda={torch.version.cuda}. "
                "Install/select a CUDA-enabled PyTorch kernel, then restart and run again."
            )
        return torch.device("cpu")
    device = torch.device(DEVICE_OVERRIDE)
    if device.type == "cuda" and not cuda_available:
        raise RuntimeError("DEVICE_OVERRIDE requests CUDA, but torch.cuda.is_available() is False.")
    return device


def make_args(model_id: str, device: torch.device) -> Namespace:
    return Namespace(
        task_name="long_term_forecast",
        is_training=0,
        model_id=model_id,
        model="GPT4TS",
        data="custom",
        root_path=str(DATA_FILE.parent) + os.sep,
        data_path=DATA_FILE.name,
        features="M",
        target=TARGET,
        freq=FREQ,
        checkpoints=str(CHECKPOINT_ROOT) + os.sep,
        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=PRED_LEN,
        seasonal_patterns="Monthly",
        inverse=False,
        mask_rate=0.25,
        anomaly_ratio=0.25,
        top_k=5,
        num_kernels=6,
        enc_in=ENC_IN,
        dec_in=ENC_IN,
        c_out=C_OUT,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        e_layers=2,
        d_layers=1,
        d_ff=D_FF,
        moving_avg=25,
        factor=1,
        distil=True,
        dropout=0.1,
        embed="timeF",
        activation="gelu",
        output_attention=False,
        num_workers=0,
        itr=1,
        train_epochs=1,
        batch_size=BATCH_SIZE,
        patience=5,
        learning_rate=0.0001,
        des="test",
        lradj="type1",
        use_amp=False,
        task_loss="l1",
        distill_loss="l1",
        logits_loss="l1",
        use_gpu=(device.type == "cuda"),
        gpu=0,
        use_multi_gpu=False,
        devices="0",
        p_hidden_dims=[128, 128],
        p_hidden_layers=2,
        tmax=20,
        cos=1,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        word_embedding_path=str(WTE_PATH),
        task_w=1.0,
        feature_w=0.01,
        logits_w=1.0,
        gpt_layers=GPT_LAYERS,
        percent=100,
        train_ratio=1.0,
        log_fine_name="notebook_eval_no_write.txt",
        noise_scale=-100,
        bootstrap_eval=0,
    )


device = resolve_device()
base_args = make_args("BTCUSDT_96_16_ori", device)
test_data, test_loader = data_provider(base_args, "test")
print("device", device)
print("test_windows_after_drop_last", len(test_loader) * BATCH_SIZE)
print("test_loader_batches", len(test_loader))


percent 100
test 14020
device cuda:0
test_windows_after_drop_last 13952
test_loader_batches 109


In [7]:
def feature_columns(csv_path: Path, target: str) -> list[str]:
    cols = list(pd.read_csv(csv_path, nrows=1).columns)
    cols.remove(target)
    cols.remove("date")
    return cols + [target]


def inverse_3d(scaler, values: np.ndarray) -> np.ndarray:
    flat = values.reshape(-1, values.shape[-1])
    restored = scaler.inverse_transform(flat)
    return restored.reshape(values.shape)


def build_window_times(csv_path: Path, seq_len: int, pred_len: int, n_windows: int) -> tuple[np.ndarray, np.ndarray]:
    """Return context and target timestamps for Dataset_Custom test windows."""
    dates = pd.to_datetime(pd.read_csv(csv_path, usecols=["date"])["date"])
    num_test = int(len(dates) * 0.2)
    test_border1 = len(dates) - num_test - seq_len
    starts = test_border1 + np.arange(n_windows)

    context_grid = starts[:, None] + np.arange(seq_len)[None, :]
    target_grid = starts[:, None] + seq_len + np.arange(pred_len)[None, :]

    context_times = dates.iloc[context_grid.reshape(-1)].to_numpy().reshape(n_windows, seq_len)
    target_times = dates.iloc[target_grid.reshape(-1)].to_numpy().reshape(n_windows, pred_len)
    return context_times, target_times


FEATURE_COLS = feature_columns(DATA_FILE, TARGET)
CLOSE_IDX = FEATURE_COLS.index(TARGET)
print(FEATURE_COLS)
print("close_idx", CLOSE_IDX)


['open', 'high', 'low', 'volume', 'quote_volume', 'number_of_trades', 'close']
close_idx 6


In [8]:
def collect_predictions(model, device: torch.device, args: Namespace, loader, max_batches: int | None = None):
    preds, trues, contexts = [], [], []
    model.eval()
    with torch.no_grad():
        for batch_idx, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            outputs = model(batch_x[:, -args.seq_len :, :])
            pred = outputs["outputs_time"][:, -args.pred_len :, :]
            true = batch_y[:, -args.pred_len :, :]
            preds.append(pred.detach().cpu().numpy())
            trues.append(true.detach().cpu().numpy())
            contexts.append(batch_x.detach().cpu().numpy())
    if not preds:
        raise RuntimeError("No prediction batches were collected.")
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0), np.concatenate(contexts, axis=0)


def summarize_predictions(preds_scaled, trues_scaled, preds_inv, trues_inv, contexts_inv) -> dict[str, float]:
    mae, mse, rmse, mape, mspe = calf_metric(preds_scaled, trues_scaled)
    close_pred = preds_inv[:, :, CLOSE_IDX]
    close_true = trues_inv[:, :, CLOSE_IDX]
    last_context_close = contexts_inv[:, -1, CLOSE_IDX]
    close_err = close_pred - close_true
    first_pred_dir = np.sign(close_pred[:, 0] - last_context_close)
    first_true_dir = np.sign(close_true[:, 0] - last_context_close)
    non_flat = first_true_dir != 0
    if np.any(non_flat):
        direction_acc = float(np.mean(first_pred_dir[non_flat] == first_true_dir[non_flat]))
    else:
        direction_acc = np.nan
    return {
        "mae_scaled": float(mae),
        "mse_scaled": float(mse),
        "rmse_scaled": float(rmse),
        "mape_scaled": float(mape),
        "mspe_scaled": float(mspe),
        "close_mae_price": float(np.mean(np.abs(close_err))),
        "close_rmse_price": float(np.sqrt(np.mean(close_err**2))),
        "close_direction_acc_h1": direction_acc,
    }


def load_checkpoint_into_model(model, checkpoint_path: Path):
    state_dict = load_state_dict_checkpoint(checkpoint_path, map_location="cpu")
    incompatible = model.load_state_dict(state_dict, strict=STRICT_LOAD)
    del state_dict
    return incompatible


In [9]:
all_rows = []
prediction_store = {}
start_time = time.time()

for row in selected.itertuples(index=False):
    print(f"\n=== {row.variant} itr={row.itr} ===")
    args = make_args(row.model_id, device)
    exp = Exp_Long_Term_Forecast(args)
    incompatible = load_checkpoint_into_model(exp.model, Path(row.checkpoint_path))
    print("load_state_dict", incompatible)

    preds_scaled, trues_scaled, contexts_scaled = collect_predictions(
        exp.model, exp.device, args, test_loader, max_batches=MAX_BATCHES
    )
    preds_inv = inverse_3d(test_data.scaler, preds_scaled).astype(np.float32)
    trues_inv = inverse_3d(test_data.scaler, trues_scaled).astype(np.float32)
    contexts_inv = inverse_3d(test_data.scaler, contexts_scaled).astype(np.float32)
    context_times, target_times = build_window_times(DATA_FILE, SEQ_LEN, PRED_LEN, preds_scaled.shape[0])

    summary = summarize_predictions(preds_scaled, trues_scaled, preds_inv, trues_inv, contexts_inv)
    summary.update(
        {
            "variant": row.variant,
            "itr": row.itr,
            "model_id": row.model_id,
            "n_windows": int(preds_scaled.shape[0]),
        }
    )
    all_rows.append(summary)

    prediction_store[(row.variant, row.itr)] = {
        "preds_scaled": preds_scaled,
        "trues_scaled": trues_scaled,
        "preds_inv": preds_inv,
        "trues_inv": trues_inv,
        "contexts_inv": contexts_inv,
        "context_times": context_times,
        "target_times": target_times,
        "checkpoint_path": Path(row.checkpoint_path),
    }

    del exp, preds_scaled, trues_scaled, contexts_scaled
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_rows).sort_values(["variant", "itr"]).reset_index(drop=True)
print(f"elapsed_sec={time.time() - start_time:.1f}")
display(summary_df)



=== dropAttn_keepWE itr=0 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_dropAttn_keepWE
load_state_dict <All keys matched successfully>

=== dropAttn_keepWE itr=1 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_dropAttn_keepWE
load_state_dict <All keys matched successfully>

=== llm_to_attn itr=0 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_llm_to_attn
load_state_dict <All keys matched successfully>

=== llm_to_attn itr=1 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_llm_to_attn
load_state_dict <All keys matched successfully>

=== llm_to_trsf itr=0 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_llm_to_trsf
load_state_dict <All keys matched successfully>

=== llm_to_trsf itr=1 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_llm_to_trsf
load_state_dict <All keys matched successfully>

=== ori itr=0 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_ori
load_state_dict <All keys matched successfully>
Orig-- BTCUSDT_96_16_ori

=== ori itr=1 ===
Use GPU: cuda:0
model_id  BTCUSDT_96_16_ori
load_state_dict <All ke

,mae_scaled,mse_scaled,rmse_scaled,mape_scaled,mspe_scaled,close_mae_price,close_rmse_price,close_direction_acc_h1,variant,itr,model_id,n_windows
0,0.169939,0.262389,0.512240,117.818367,1910.910278,471.655823,731.324036,0.523055,dropAttn_keepWE,0,BTCUSDT_96_16_dropAttn_keepWE,13952
1,0.169251,0.261371,0.511244,116.956268,1932.431152,472.208862,730.961426,0.513876,dropAttn_keepWE,1,BTCUSDT_96_16_dropAttn_keepWE,13952
2,0.170279,0.266996,0.516717,118.104912,1884.996704,483.060425,744.704407,0.516458,llm_to_attn,0,BTCUSDT_96_16_llm_to_attn,13952
3,0.170846,0.263922,0.513734,114.905922,1555.473145,477.924896,739.022949,0.516458,llm_to_attn,1,BTCUSDT_96_16_llm_to_attn,13952
4,0.171636,0.269421,0.519057,114.660896,1705.366455,517.226562,791.167786,0.510075,llm_to_trsf,0,BTCUSDT_96_16_llm_to_trsf,13952
5,0.169696,0.265212,0.514987,120.698730,2546.892578,474.950836,732.437988,0.512513,llm_to_trsf,1,BTCUSDT_96_16_llm_to_trsf,13952
6,0.168834,0.267050,0.516769,114.806938,1656.743896,512.027039,783.856506,0.512083,ori,0,BTCUSDT_96_16_ori,13952
7,0.168004,0.266842,0.516568,113.636490,1618.877319,496.250000,766.587341,0.523557,ori,1,BTCUSDT_96_16_ori,13952


In [10]:
# Compare with the author's saved summary file if present.
result_file = CALF_ROOT / "BTCUSDT_15m_result.txt"
if result_file.exists():
    print(result_file)
    print(result_file.read_text(encoding="utf-8", errors="replace"))
else:
    print("No BTCUSDT_15m_result.txt found.")


C:\Users\USER\Desktop\time_series\LLM for timeseries\LLMsForTimeSeries\CALF\BTCUSDT_15m_result.txt
BTCUSDT_96_16_ori
mae:0.16842 , mae_std:0.000415 , mse:0.26695 , mse_std:0.000104
BTCUSDT_96_16_dropAttn_keepWE
mae:0.16959 , mae_std:0.000344 , mse:0.26188 , mse_std:0.000509
BTCUSDT_96_16_llm_to_attn
mae:0.17056 , mae_std:0.000284 , mse:0.26546 , mse_std:0.001537
BTCUSDT_96_16_llm_to_trsf
mae:0.17067 , mae_std:0.00097 , mse:0.26732 , mse_std:0.002104
BTCUSDT_96_32_ori
mae:0.18722 , mae_std:0.002459 , mse:0.29046 , mse_std:0.003274
BTCUSDT_96_32_dropAttn_keepWE
mae:0.18644 , mae_std:0.001557 , mse:0.28703 , mse_std:0.002701
BTCUSDT_96_32_llm_to_attn
mae:0.1865 , mae_std:0.000162 , mse:0.28817 , mse_std:0.000123
BTCUSDT_96_32_llm_to_trsf
mae:0.1866 , mae_std:0.000783 , mse:0.28873 , mse_std:0.000664
BTCUSDT_96_96_ori
mae:0.21606 , mae_std:0.000641 , mse:0.32962 , mse_std:0.001445
BTCUSDT_96_96_dropAttn_keepWE
mae:0.21529 , mae_std:0.000337 , mse:0.32634 , mse_std:0.000307
BTCUSDT_96_96_ll

In [11]:
def window_close_error(store_item) -> np.ndarray:
    pred = store_item["preds_inv"][:, :, CLOSE_IDX]
    true = store_item["trues_inv"][:, :, CLOSE_IDX]
    return np.mean(np.abs(pred - true), axis=1)


def choose_windows(store_item) -> dict[str, int]:
    err = window_close_error(store_item)
    return {
        "first": 0,
        "median_error": int(np.argsort(err)[len(err) // 2]),
        "worst_error": int(np.argmax(err)),
        "last": len(err) - 1,
    }


def assert_predictions_ready():
    if not prediction_store:
        raise RuntimeError("prediction_store is empty. Run the checkpoint evaluation cell before plotting.")


def plot_context_ground_truth_forecast(key, window_idx: int, feature: str = TARGET):
    assert_predictions_ready()
    item = prediction_store[key]
    feature_idx = FEATURE_COLS.index(feature)

    context_times = item["context_times"][window_idx]
    target_times = item["target_times"][window_idx]
    context = item["contexts_inv"][window_idx, :, feature_idx]
    true = item["trues_inv"][window_idx, :, feature_idx]
    pred = item["preds_inv"][window_idx, :, feature_idx]

    true_times = np.concatenate([[context_times[-1]], target_times])
    pred_times = np.concatenate([[context_times[-1]], target_times])
    true_line = np.concatenate([[context[-1]], true])
    pred_line = np.concatenate([[context[-1]], pred])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(context_times, context, color="0.35", linewidth=1.8, label=f"history {SEQ_LEN}")
    ax.plot(true_times, true_line, color="tab:green", marker="o", linewidth=2.0, label=f"ground truth next {PRED_LEN}")
    ax.plot(pred_times, pred_line, color="tab:red", marker="o", linewidth=2.0, label=f"forecast {key[0]} itr={key[1]}")
    ax.axvline(context_times[-1], color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_title(f"{feature}: 96-step context + 16-step forecast | window={window_idx}")
    ax.set_xlabel("time")
    ax.set_ylabel(feature)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()
    return fig, ax


example_key = (SELECTED_VARIANTS[0], SELECTED_ITERS[0])
if example_key not in prediction_store:
    assert_predictions_ready()
    example_key = next(iter(prediction_store))

chosen_windows = choose_windows(prediction_store[example_key])
chosen_windows


{'first': 0, 'median_error': 11346, 'worst_error': 6149, 'last': 13951}

In [12]:
for label, idx in chosen_windows.items():
    print(label, idx)
    plot_context_ground_truth_forecast(example_key, idx, TARGET)


first 0
median_error 11346
worst_error 6149
last 13951


In [18]:
def plot_variant_comparison_with_context(window_idx: int, itr: int = 0, feature: str = TARGET):
    assert_predictions_ready()
    feature_idx = FEATURE_COLS.index(feature)

    available_keys = [(variant, itr) for variant in SELECTED_VARIANTS if (variant, itr) in prediction_store]
    if not available_keys:
        raise RuntimeError(f"No predictions available for itr={itr}.")

    base_item = prediction_store[available_keys[0]]
    context_times = base_item["context_times"][window_idx]
    target_times = base_item["target_times"][window_idx]
    context = base_item["contexts_inv"][window_idx, :, feature_idx]
    true = base_item["trues_inv"][window_idx, :, feature_idx]
    future_times = np.concatenate([[context_times[-1]], target_times])
    true_line = np.concatenate([[context[-1]], true])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(context_times, context, color="0.35", linewidth=1.8, label=f"history {SEQ_LEN}")
    ax.plot(future_times, true_line, color="black", marker="o", linewidth=2.4, label=f"ground truth next {PRED_LEN}")

    for key in available_keys:
        item = prediction_store[key]
        pred = item["preds_inv"][window_idx, :, feature_idx]
        pred_line = np.concatenate([[context[-1]], pred])
        ax.plot(future_times, pred_line, marker="o", linewidth=1.7, label=key[0])

    ax.axvline(context_times[-1], color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_title(f"Ablation comparison with context | itr={itr}, window={window_idx}, feature={feature}")
    ax.set_xlabel("time")
    ax.set_ylabel(feature)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()
    return fig, ax

plot_variant_comparison_with_context(chosen_windows["median_error"], itr=SELECTED_ITERS[0], feature=TARGET)
plot_variant_comparison_with_context(chosen_windows["worst_error"], itr=SELECTED_ITERS[0], feature=TARGET)


(<Figure size 1400x500 with 1 Axes>,
 <Axes: title={'center': 'Ablation comparison with context | itr=0, window=6149, feature=close'}, xlabel='time', ylabel='close'>)

In [19]:
def plot_horizon_mae(feature: str = TARGET):
    feature_idx = FEATURE_COLS.index(feature)
    plt.figure(figsize=(12, 5))
    horizons = np.arange(1, PRED_LEN + 1)
    for key, item in prediction_store.items():
        pred = item["preds_inv"][:, :, feature_idx]
        true = item["trues_inv"][:, :, feature_idx]
        mae_by_horizon = np.mean(np.abs(pred - true), axis=0)
        plt.plot(horizons, mae_by_horizon, marker="o", label=f"{key[0]} itr={key[1]}")
    plt.title(f"{feature} MAE by forecast horizon")
    plt.xlabel("horizon step")
    plt.ylabel("MAE in original scale")
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.show()

plot_horizon_mae(TARGET)


In [20]:
def plot_feature_horizon_heatmap(key):
    item = prediction_store[key]
    abs_err = np.abs(item["preds_inv"] - item["trues_inv"])
    heatmap = np.mean(abs_err, axis=0).T  # feature x horizon
    plt.figure(figsize=(12, 4.5))
    im = plt.imshow(heatmap, aspect="auto", cmap="magma")
    plt.colorbar(im, label="MAE in original scale")
    plt.yticks(np.arange(len(FEATURE_COLS)), FEATURE_COLS)
    plt.xticks(np.arange(PRED_LEN), np.arange(1, PRED_LEN + 1))
    plt.xlabel("horizon step")
    plt.title(f"Feature x horizon absolute error: {key[0]} itr={key[1]}")
    plt.tight_layout()
    plt.show()

plot_feature_horizon_heatmap(example_key)


In [21]:
def plot_prediction_scatter(key, feature: str = TARGET, sample_points: int = 5000):
    item = prediction_store[key]
    feature_idx = FEATURE_COLS.index(feature)
    true = item["trues_inv"][:, :, feature_idx].reshape(-1)
    pred = item["preds_inv"][:, :, feature_idx].reshape(-1)
    if len(true) > sample_points:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(true), size=sample_points, replace=False)
        true = true[idx]
        pred = pred[idx]
    lo = min(true.min(), pred.min())
    hi = max(true.max(), pred.max())
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(true, pred, s=8, alpha=0.35)
    plt.plot([lo, hi], [lo, hi], color="black", linewidth=1)
    plt.xlabel("actual")
    plt.ylabel("predicted")
    plt.title(f"Prediction scatter: {key[0]} itr={key[1]}, {feature}")
    plt.tight_layout()
    plt.show()

plot_prediction_scatter(example_key, TARGET)


## Notes

- `mae_scaled` and `mse_scaled` should be the closest values to the author's text summary because CALF evaluates on standardized data.
- Price-scale metrics are easier to interpret, but they are computed after inverse-transforming the standardized outputs.
- The test loader follows CALF's default `drop_last=True`; this keeps metric behavior close to the author's `test()` loop but skips the final incomplete batch.
- The line charts now show the exact 96-step model input context followed by the 16-step ground truth and forecast. The vertical dashed line marks the forecast start.
- The notebook mirrors CALF's `is_training=0` inference path by using the same `data_provider(..., "test")`, model config, `model.eval()`, `torch.no_grad()`, and `outputs["outputs_time"][:, -pred_len:, :]` slice. It explicitly loads the selected checkpoint because the current local `Exp_Long_Term_Forecast.test(..., test=1)` source has checkpoint loading commented out.
- If a checkpoint does not load with `STRICT_LOAD=True`, inspect the error before changing it to `False`; mismatched keys usually mean the runtime config does not match the checkpoint architecture.
